# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
df.shape

(891, 15)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


## A.2. Missing values & Duplicate data

In [9]:
print('số lượng giá trị thiếu:\n', df.isnull().sum())
print('số lượng hàng trùng lặp:\n', df.duplicated().sum())




số lượng giá trị thiếu:
 survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
số lượng hàng trùng lặp:
 107


## A.3. Invalid values

In [10]:
df.describe()



,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [11]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df[['sibsp', 'parch', 'family_size']].head(10)


,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1
5,0,0,1
6,0,0,1
7,3,1,5
8,0,2,3
9,1,0,2


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [12]:
df[['age', 'fare']].mean()


age     29.699118
fare    32.204208
dtype: float64

In [13]:
df[['age', 'fare']].median()

age     28.0000
fare    14.4542
dtype: float64

In [14]:
df[['age', 'fare', 'pclass', 'sex']].mode()

,age,fare,pclass,sex
0,24.0,8.05,3,male


## Group 2 — Dispersion

In [15]:
df[['age', 'fare']].std()


age     14.526497
fare    49.693429
dtype: float64

In [16]:
df[['age', 'fare']].var()

age      211.019125
fare    2469.436846
dtype: float64

In [17]:
R_age = df['age'].max() - df['age'].min()
R_fare = df['fare'].max() - df['fare'].min()
print('Range age:', R_age)
print('Range fare:', R_fare)

Range age: 79.58
Range fare: 512.3292


## Group 3 — Location and Shape

In [18]:
print('độ lệch :\n',df[['age', 'fare']].skew())
print('độ nhọn :\n',df[['age', 'fare']].kurt())


độ lệch :
 age     0.389108
fare    4.787317
dtype: float64
độ nhọn :
 age      0.178274
fare    33.398141
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [19]:
sv = df.groupby('pclass')['survived'].mean() * 100
print('Tỷ lệ sống sót theo hạng vé:\n', sv)
cl = sv.max()- sv.min()
print('Chênh lệch tỷ lệ sống sót giữa hạng vé cao nhất và thấp nhất:', cl)


Tỷ lệ sống sót theo hạng vé:
 pclass
1    62.962963
2    47.282609
3    24.236253
Name: survived, dtype: float64
Chênh lệch tỷ lệ sống sót giữa hạng vé cao nhất và thấp nhất: 38.726710417138115


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [20]:
df.groupby(['sex', 'pclass'])['survived'].mean() * 100


sex     pclass
female  1         96.808511
        2         92.105263
        3         50.000000
male    1         36.885246
        2         15.740741
        3         13.544669
Name: survived, dtype: float64

Phụ nữ có tỷ lệ sống cao hơn nam giới rất nhiều ở mọi hạng vé => giới tính ảnh hưởng mạnh mẽ nhất, tuy nhiên hạng vé cũng quan trọng vì nam giới có hạng vé 1 sống sót cao hơn nam giới có hạng vé 3

## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [21]:
df.groupby('survived')['fare'].mean()


survived
0    22.117887
1    48.395408
Name: fare, dtype: float64

Có, những khách hàng sống sót có giá vé trung bình cao hơn gấp đôi so với những khách không sống sót

## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [22]:
df.groupby('family_size')['survived'].mean() * 100


family_size
1     30.353818
2     55.279503
3     57.843137
4     72.413793
5     20.000000
6     13.636364
7     33.333333
8      0.000000
11     0.000000
Name: survived, dtype: float64

Có, hàng khách đi cùng gia đình từ 2 đến 4 người có tỷ lệ sống sót cao nhất (55-73%), những khách hàng đi một mình hoặc cùng gia đình từ 5 người trở lên có tỷ lệ sống sót thấp hơn 

## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [ ]:
df.groupby('embark_town')['survived'].mean() * 100

Cảng cherbourg có tỷ lệ sống sót cao nhất

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

*(Viết insight của bạn vào đây...)*

Trong thảm kịch giới tính nữ có tỷ lệ sống sót vượt trội so với nam giới ỏ bất kỳ hạng vé nào
Khách hàng có vé hạng một hoặc trả giá vé cao có tỷ lệ sống sót cao rất nhiều lần
Việc có người thân đi cùng cũng mang lại lợi thế sinh tồn, tuy nhiên chỉ đúng với các gia đình từ 2 đến 4 người. Những người đi 1 mình hoặc gia đình qua đông đều gặp bất lợi trong việc sơ tán
